In [11]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from sentence_transformers import util

EMB_DIR = Path("../data/embeddings")
OUTPUT_DIR = Path("../data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample = pd.read_csv(EMB_DIR / "cicids_sample.csv", index_col="sample_id", low_memory=False)
embeddings = np.load(EMB_DIR / "cicids_embeddings.npy")
assert len(sample) == len(embeddings), "Embedding/CSV length mismatch"
print(f"Loaded {len(sample)} samples | Shape: {embeddings.shape}")

Loaded 10684 samples | Shape: (10684, 384)


In [12]:
# Community Detection
print("Running community detection on embedding space...")
communities = util.community_detection(
    embeddings,
    min_community_size=3,
    threshold=0.75,
)
print(f"Found {len(communities)} semantic communities")

Running community detection on embedding space...
Found 19 semantic communities


In [13]:
# Assign community IDs back to samples
community_assignments = np.full(len(sample), -1, dtype=int)
for idx, comm_indices in enumerate(communities):
    for sample_idx in comm_indices:
        community_assignments[sample_idx] = idx

valid_mask = community_assignments != -1
community_df = sample[valid_mask].copy()
community_df["community_id"] = community_assignments[valid_mask]
community_df.to_csv(OUTPUT_DIR / "community_assignments.csv", index=False)
print(f"Assigned {len(community_df)} alerts to {community_df['community_id'].nunique()} communities")

# Quick sanity check
print("\nCommunity composition preview:")
for cid in sorted(community_df['community_id'].unique())[:5]:
    group = community_df[community_df['community_id'] == cid]
    dominant_label = group['Label'].mode().iloc[0]
    dominant_tactic = group['attck_tactic'].mode().iloc[0]
    print(f"  Community {cid}: {len(group)} alerts | {dominant_label} | {dominant_tactic}")

Assigned 10684 alerts to 19 communities

Community composition preview:
  Community 0: 8665 alerts | PortScan | Impact
  Community 1: 1013 alerts | FTP Patator | Credential Access
  Community 2: 329 alerts | BENIGN | Benign
  Community 3: 245 alerts | BENIGN | Benign
  Community 4: 148 alerts | FTP Patator | Credential Access


In [14]:
# LLM initialization
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=20,    
    n_threads=8,
    n_batch=256,
    verbose=False,
    seed=42,
)
print(f"LLM loaded: {MODEL_PATH}")

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM loaded: ../models/qwen2.5-3b-instruct-q4_k_m.gguf


In [15]:
# Define JSON schema for triple extraction
SECURITY_RELATIONS = [
    "PERFORMS_RECONNAISSANCE",
    "BRUTE_FORCES_CREDENTIAL",
    "EXPLOITS_VULNERABILITY",
    "ESTABLISHES_C2",
    "CAUSES_DENIAL_OF_SERVICE",
    "MOVES_LATERALLY",
    "EXFILTRATES_DATA",
    "EXECUTES_PAYLOAD",
]

TRIPLE_SCHEMA = {
    "type": "object",
    "properties": {
        "triples": {
            "type": "array",
            "minItems": 4,
            "maxItems": 4,
            "items": {
                "type": "object",
                "properties": {
                    "subject": {
                        "type": "string"
                        # Open: LLM names the actor it observes in the alert text
                    },
                    "relation": {
                        "type": "string",
                        "enum": SECURITY_RELATIONS
                        # Constrained: grammar enforces one of the 8 defined relations
                    },
                    "target": {
                        "type": "string"
                        # Open: LLM names the target resource it observes
                    }
                },
                "required": ["subject", "relation", "target"]
            }
        }
    },
    "required": ["triples"]
}

grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
print(f"Grammar initialized with {len(SECURITY_RELATIONS)} security relations")
print(f"Ontology: {SECURITY_RELATIONS}")

Grammar initialized with 8 security relations
Ontology: ['PERFORMS_RECONNAISSANCE', 'BRUTE_FORCES_CREDENTIAL', 'EXPLOITS_VULNERABILITY', 'ESTABLISHES_C2', 'CAUSES_DENIAL_OF_SERVICE', 'MOVES_LATERALLY', 'EXFILTRATES_DATA', 'EXECUTES_PAYLOAD']


In [16]:
RELATION_GUIDE = """
Relation definitions (use exactly as written):
  PERFORMS_RECONNAISSANCE  : active discovery, port scanning, or service enumeration
  BRUTE_FORCES_CREDENTIAL  : repeated authentication attempts against a service
  EXPLOITS_VULNERABILITY   : exploitation of a software or configuration flaw
  ESTABLISHES_C2           : outbound communication to a command-and-control channel
  CAUSES_DENIAL_OF_SERVICE : flooding or resource exhaustion of a target service
  MOVES_LATERALLY          : accessing internal hosts after initial compromise
  EXFILTRATES_DATA         : transferring data out of the environment
  EXECUTES_PAYLOAD         : running code or commands on a target
""".strip()


def normalise_entity(text: str) -> str:
    """Lowercase, strip, replace spaces and special characters with underscores.
    Keeps entity names consistent across communities for graph node deduplication."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text[:80]  # cap length to prevent runaway entity names


def extract_triples(alert_texts: list, community_id: int) -> list:
    block = "\n".join([f"- {t}" for t in alert_texts])

    prompt = f"""[INST] You are a cybersecurity analyst performing threat analysis.
Read the network alerts below and extract exactly 4 semantic triples that describe
the attack behaviour using standard security terminology.

{RELATION_GUIDE}

For each triple:
- subject : name the specific actor or source you observe in the text
            (e.g. "ssh_scanner", "ftp_brute_force_client", "botnet_node")
- relation: choose the ONE relation from the list above that best fits
- target  : name the specific service, port, or resource being acted on
            (e.g. "ssh_port_22", "ftp_authentication_service", "http_web_server")

Name what you actually observe. Do not use generic placeholders like "source" or "destination".

Example output for SSH brute-force alerts:
{{"triples": [
  {{"subject": "ssh_brute_force_client",      "relation": "BRUTE_FORCES_CREDENTIAL",  "target": "ssh_port_22_service"}},
  {{"subject": "ssh_brute_force_client",      "relation": "PERFORMS_RECONNAISSANCE",  "target": "ssh_authentication_endpoint"}},
  {{"subject": "credential_guessing_process", "relation": "EXECUTES_PAYLOAD",         "target": "password_spray_module"}},
  {{"subject": "attacker",                    "relation": "BRUTE_FORCES_CREDENTIAL",  "target": "user_account_store"}}
]}}

ALERTS (Community {community_id}):
{block}

Return ONLY valid JSON. [/INST]"""

    out = llm(
        prompt,
        max_tokens=512,
        temperature=0,
        seed=42,
        grammar=grammar,
        repeat_penalty=1.1,
        stop=["[/INST]"]
    )
    raw = out["choices"][0]["text"].strip()

    try:
        parsed  = json.loads(raw)
        triples = parsed.get("triples", [])
    except Exception as e:
        print(f"  [Community {community_id}] JSON parse failed: {e}")
        triples = []

    # Validate structure and normalise entity names
    valid_relations = set(SECURITY_RELATIONS)
    validated = []

    for t in triples:
        subj = normalise_entity(str(t.get("subject", "")))
        rel  = str(t.get("relation", "")).upper().strip()
        tgt  = normalise_entity(str(t.get("target", "")))

        # Skip triples with empty fields
        if not subj or not tgt:
            continue
        # Fall back to PERFORMS_RECONNAISSANCE if relation is outside enum
        # (the grammar should prevent this, but we guard defensively)
        if rel not in valid_relations:
            rel = "PERFORMS_RECONNAISSANCE"

        validated.append({"subject": subj, "relation": rel, "target": tgt})

    # Pad to 4 if the LLM returned fewer than expected (fallback only)
    fallback = {
        "subject":  f"community_{community_id}_actor",
        "relation": "PERFORMS_RECONNAISSANCE",
        "target":   f"community_{community_id}_target"
    }
    validated = (validated + [fallback] * 4)[:4]
    return validated

In [17]:
# Warmup and extraction loop
print("Warming up LLM")
_ = llm("[INST] Test. [/INST]", max_tokens=5, temperature=0, seed=42)
print("Starting extraction\n")

community_triples = {}
for cid, group in community_df.groupby("community_id"):
    # Take up to 6 representative alerts per community
    texts = group["alert_text"].dropna().head(6).tolist()
    triples = extract_triples(texts, int(cid))
    community_triples[str(int(cid))] = triples

    # Live inspection
    dominant_label = group['Label'].mode().iloc[0]
    print(f"Community {cid} [{dominant_label}]:")
    for t in triples:
        print(f"  {t['subject']} --[{t['relation']}]-- {t['target']}")
    print()

print(f"Extraction complete: {len(community_triples)} communities processed")

Warming up LLM
Starting extraction

Community 0 [PortScan]:
  ftp_brute_force_client --[BRUTE_FORCES_CREDENTIAL]-- ftp_authentication_service
  http_scanner --[PERFORMS_RECONNAISSANCE]-- http_web_server
  http_scanner --[ESTABLISHES_C2]-- command_and_control_channel
  http_scanner --[EXFILTRATES_DATA]-- internal_data_store

Community 1 [FTP Patator]:
  ssh_scanner --[PERFORMS_RECONNAISSANCE]-- ssh_port_22
  ftp_brute_force_client --[BRUTE_FORCES_CREDENTIAL]-- ftp_authentication_service
  botnet_node --[EXPLOITS_VULNERABILITY]-- unpatched_ssh_server
  attacker --[ESTABLISHES_C2]-- c2_server

Community 2 [BENIGN]:
  dns_brute_force_client --[BRUTE_FORCES_CREDENTIAL]-- dns_service
  dns_brute_force_client --[PERFORMS_RECONNAISSANCE]-- dns_service
  attacker --[EXPLOITS_VULNERABILITY]-- dns_service
  attacker --[ESTABLISHES_C2]-- unknown

Community 3 [BENIGN]:
  dns_probe_client --[ESTABLISHES_C2]-- dns_service
  dns_probe_client --[PERFORMS_RECONNAISSANCE]-- dns_service
  attacker --[ESTA

In [18]:
# Metrics and output
all_subjects    = set()
all_targets     = set()
relation_counts = Counter()
valid_count     = 0
total_count     = 0

for triples in community_triples.values():
    for t in triples:
        total_count += 1
        if all(k in t and t[k] for k in ["subject", "relation", "target"]):
            valid_count += 1
            all_subjects.add(t["subject"])
            all_targets.add(t["target"])
            relation_counts[t["relation"]] += 1

triple_metrics = {
    "n_communities":         len(community_triples),
    "total_triples":         total_count,
    "valid_triples":         valid_count,
    "valid_ratio":           round(valid_count / max(total_count, 1), 4),
    "unique_subjects":       len(all_subjects),
    "unique_targets":        len(all_targets),
    "unique_entities_total": len(all_subjects | all_targets),
    "relation_counts":       dict(relation_counts),
    "relation_coverage":     f"{len(relation_counts)}/{len(SECURITY_RELATIONS)} ontology relations used",
}

with open(OUTPUT_DIR / "triple_metrics.json", "w") as f:
    json.dump(triple_metrics, f, indent=2)

with open(OUTPUT_DIR / "community_triples.json", "w", encoding="utf-8") as f:
    json.dump(community_triples, f, indent=2)

print("TRIPLE EXTRACTION METRICS")
for k, v in triple_metrics.items():
    print(f"  {k}: {v}")

print(f"\nAll unique subjects discovered: {sorted(all_subjects)}")
print(f"\nAll unique targets discovered:  {sorted(all_targets)}")
print(f"\nOutputs saved to: {OUTPUT_DIR}")

TRIPLE EXTRACTION METRICS
  n_communities: 19
  total_triples: 76
  valid_triples: 76
  valid_ratio: 1.0
  unique_subjects: 23
  unique_targets: 34
  unique_entities_total: 49
  relation_counts: {'BRUTE_FORCES_CREDENTIAL': 13, 'PERFORMS_RECONNAISSANCE': 22, 'ESTABLISHES_C2': 22, 'EXFILTRATES_DATA': 5, 'EXPLOITS_VULNERABILITY': 10, 'EXECUTES_PAYLOAD': 3, 'MOVES_LATERALLY': 1}
  relation_coverage: 7/8 ontology relations used

All unique subjects discovered: ['attacker', 'botnet_node', 'credential_guessing_process', 'dns_brute_force_client', 'dns_client', 'dns_probe_client', 'dns_query_client', 'dns_scanner', 'ftp_brute_force_client', 'http_alt', 'http_alt_client', 'http_scanner', 'https_client', 'rdp_client', 'ssh_brute_force_client', 'ssh_scanner', 'unknown_port_1098', 'unknown_port_1723', 'unknown_port_3289', 'unknown_port_4321', 'unknown_port_52822', 'unknown_port_5353', 'unknown_port_6003']

All unique targets discovered:  ['c2_server', 'command_and_control_channel', 'dns_protocol', 